# Análisis avanzado: mapa y comparativa con el INE

Requiere haber ejecutado antes `python scripts/fetch_open_datasets.py`.

**Nota importante**: el ranking por ciudad (tu snapshot) y la evolución del INE se muestran **por separado**, nunca superpuestos en el mismo eje — el IPV del INE es un índice de crecimiento desde un año base, no un precio absoluto en €, así que no es directamente comparable contra tu precio/m². Mezclarlos en un mismo gráfico sería engañoso.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.viz.plots import plot_mapa_precio_m2, plot_ranking_ciudades, plot_ipv_evolucion
from src.data.ine_ipv import obtener_evolucion_por_region

df = pd.read_csv("../data/processed/open_datasets_clean.csv", parse_dates=["fecha_publicacion"])
df.head()

## Mapa de alquiler (Madrid + Alicante)

In [ ]:
alquiler_geo = df[(df["tipo_operacion"] == "alquiler") & df["latitud"].notna()]
plot_mapa_precio_m2(alquiler_geo, titulo="Alquiler: precio/m² por ubicación")

## Ranking de ciudades (venta, tu snapshot)

In [ ]:
from scripts.analisis_avanzado import CIUDAD_A_CCAA

venta = df[df["tipo_operacion"] == "venta"].copy()
venta["comunidad_autonoma"] = venta["ciudad"].map(CIUDAD_A_CCAA)
media_propia = venta["precio_m2"].mean()
por_ciudad = venta.groupby("ciudad").agg(
    precio_m2_medio=("precio_m2", "mean"),
    comunidad_autonoma=("comunidad_autonoma", "first"),
    n_anuncios=("precio_m2", "count"),
).reset_index()
por_ciudad["indice_propio"] = por_ciudad["precio_m2_medio"] / media_propia * 100
plot_ranking_ciudades(por_ciudad)

## Evolución oficial del INE para esas mismas comunidades (contexto, no comparación directa)

In [ ]:
regiones = por_ciudad.groupby("comunidad_autonoma")["n_anuncios"].sum().sort_values(ascending=False).head(8).index.tolist()
ine = obtener_evolucion_por_region(regiones)
plot_ipv_evolucion(ine)